# 01 — Préparation et exploration

SAIM Bahaeddine et OUNI Hazem. Corpus : archives CFPB 2025. Ce notebook explore une démonstration réelle limitée aux 3 000 premières lignes de chaque export 2025. Aucun résultat sur le corpus complet ou sur 2026 n’est déduit de cet échantillon. Préparation reproductible : `python -m scripts.prepare_demo`. Pour le corpus complet : `python run_pipeline.py`.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
assert (ROOT / "src").is_dir(), "Ouvrir depuis la racine ou notebooks/"
sys.path.insert(0, str(ROOT))
import json
import sqlite3
from contextlib import closing

import pandas as pd

REPORTS = ROOT / "reports/final"
DATABASE = ROOT / "data/processed/demo.sqlite"
assert DATABASE.is_file(), "Executer python -m scripts.prepare_demo depuis la racine"

## Origine et sélection
Les récits sont conservés avec leur ID et fichier source. Les dates invalides, textes vides et doublons normalisés sont exclus. La partition 2025 est déterminée par hash (environ 80 % train, 20 % validation). Voir le README pour les conditions de réutilisation.

In [2]:
quality = json.loads((REPORTS / "data_quality.json").read_text(encoding="utf-8"))
pd.DataFrame([quality["counts"]]).T.rename(columns={0: "effectif"})

,effectif
rows_read,18000
empty_narrative,14110
kept,2521
duplicate_text,1369


In [3]:
with closing(sqlite3.connect(DATABASE)) as db:
    distribution = pd.read_sql_query("SELECT product, split, COUNT(*) AS n FROM complaints GROUP BY product, split", db)
distribution.pivot(index="product", columns="split", values="n").fillna(0).astype(int)

split,train,validation
product,,
Checking or savings account,111,26
Credit card,106,20
Credit reporting or other personal consumer reports,1336,336
Debt collection,235,62
Debt or credit management,10,3
"Money transfer, virtual currency, or money service",55,17
Mortgage,40,10
"Payday loan, title loan, personal loan, or advance loan",15,6
Prepaid card,18,4


In [4]:
pd.DataFrame(quality["lengths"])

,split,min,mean,max
0,train,51,1331.238142,35581
1,validation,51,1448.764588,34802


Les textes sont déséquilibrés et les premières lignes des fichiers ne constituent pas un échantillon représentatif. Les quasi-doublons ne sont pas détectés. La longueur et les effectifs par classe doivent guider les expériences suivantes.